# FoBench Basics
Welcome to the tutorial of the basic functionalities of FoBench. This notebook will guide you through the loading of data, preprocessing steps and some visuals. Of course we can not go through every method in detail, so feel free to dive into the documentation if you want to know more. Here and there we will hint to more functionality to discover but for now we will stick to the basics.

A note about plotting: 
Most methods have two plotting modes: matplotlib (plot_mode='mpl') and PyQtGraph (plot_mode='pyqt'), the standard in FoBench is using PyQtGraph as it performs better when it comes to larger matrix plots. However it is not very stable when working in a jupyter notebook, it should however work from other IDEs or simple scripts. If you want to know more on how to navigate FoBench plots, check out the page on documentation on Visualiasations.


In [1]:
import os
os.environ['PYQTGRAPH_QT_LIB'] = 'PyQt5'
from fobench import Fiber
%gui qt

The workhorse of FoBench is the `Fiber` class, it contains the actual fiber optic record with all its metadata and provides a lot of functionality to manipulate data in space and time. Let us go right ahead and load a file. We simply give the path to the file and we let FoBench know the manufacturer of the interrogator we used. In our case we will use data recorded on an Aragon Photonics HDAS system. You can find the supported data formats and corresponding keywords in the documentation.

In [7]:
das = Fiber('./example_data/aragon_h5/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5', 'aragon')

Read File ✓: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  8.24it/s]


Lets first have a look at the most important metadata features of the file we loaded.

In [ ]:
print(das)
# das.metadata() # for the full metadata

We can see that the file contains a single minute of strain data, lets concatenate a second file and convert into strain-rate after. 

In [9]:
das += Fiber('./example_data/aragon_h5/2024_07_11_05h16m16s_HDAS_2DRawData_Strain.h5', 'aragon') # the += syntax just calls Fiber.concatenate()
das.differentiate() # Fiber.integrate for integration of the data

Reading Aragon Photonics HDF5 file:   0%|                                                                                                                               | 0/1 [00:00<?, ?it/s]

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = './example_data/aragon_h5/2024_07_11_05h16m16s_HDAS_2DRawData_Strain.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

Our original sampling frequency is a bit too high, we will decimate the data to 50 Hz.

If we then check the aquisition parameters again, we can see that we now have two minutes of strain-rate data at 50 Hz

In [6]:
new_freq = 50
das.decimate(new_freq)
print(das)

Instance of Fiber class
recording parameters:
-----------------------------------------------------------------
units                     = strain-rate
start_time                = 2024-07-11T05:15:15.791536Z
end_time                  = 2024-07-11T05:17:15.791002Z
num_points                = 15000
total_channels            = 501
spatial_interval          = 1.0
sampling_frequency        = 50.0
gauge_length              = 6.0


The `Fiber` class stores the actual data in the `Fiber.data` attribute. This way we can easily access and extract it:

In [9]:
print(type(das.data), das.data.shape, das.data)
#das.get_data() # returns full data or data of a specific channel
#das.times # returns the time stamps for each sample in specified format

<class 'numpy.ndarray'> (15000, 178) [[ 1.18834804e-06 -6.37684889e-07 -2.83725679e-07 ... -1.46813640e-06
  -2.67803036e-06  2.14337543e-06]
 [ 1.06563654e-06 -2.16246602e-06  1.65240231e-06 ... -1.62643239e-06
  -2.79495789e-06  2.19883991e-06]
 [ 1.11171888e-06 -2.36149614e-06  1.37593010e-06 ... -1.63399324e-06
  -2.73485132e-06  2.09741189e-06]
 ...
 [-1.77894464e-07  3.84527275e-04  3.44950141e-05 ...  1.18334423e-06
  -3.66664220e-06  3.90184058e-06]
 [-2.14032917e-08  3.84514905e-04  3.46258142e-05 ...  9.60871786e-07
  -4.00273815e-06  4.05944353e-06]
 [-1.37796360e-07  3.84627386e-04  3.44931464e-05 ...  1.29725038e-06
  -4.20291677e-06  4.28760489e-06]]


Most operations on the data are done inplace. Just in case we mess something up later on, at any point we can get a copy of the records current state with `Fiber.copy()`

In [25]:
backup_das = das.copy()

Before we have a look at the data, we should apply some basic preprocessing. Let us start with a simple bandpass filter between 0.1 and 20 Hz. We can check if the filter was applied successfully by looking at the list stored in `Fiber.processing`:

In [10]:
das.filter(f_type='bandpass', freq=(0.1, 20)) # other filter options are highpass, lowpass and bandstop. Fobench also provides more specialized filters such as median, Cheby and FIR-filters
das.processing

[{'instance creation': 'Mon Aug 31 20:22:05 2026'},
 {'differentiate': {'method': 'gradient', 'dim': 't'}},
 {'decimate': {'new_freq': 50, 'f_type': 'fir-remez'}},
 {'preprocess': {'alpha': 0.05,
   'order': 1,
   'sym': True,
   'axis': 0,
   'steps': (True, True, True)}},
 {'filter': {'f_type': 'bandpass',
   'freq': (0.1, 20),
   'pre_process': True,
   'alpha': 0.05,
   'order': 1,
   'sym': True,
   'options': {}}}]

The list keeps track of all preprocessing steps included in FoBench so that at any point we can go back and check which steps we took in what order. This is done by storing the methods name and all parameters it is called with.

We can see that before bandpass filtering the data was preprocessed, in FoBench this mean demeaning, detrending and tapering. The filter function by default conveniently performs these steps before the actual filtering. Of course all these methods can be used individually and `filter` can be used with the `pre_process` parameter set to `False`.

Note also that most methods have additional parameters such as the removal of higher order polynomial trends by passing the `order` to `Fiber.detrend()`. A lot of methods can be applied in both space and time, this is determined by the `dim` parameter.

Now back to our data! Let us plot the record in both time and frequency domain:

In [12]:
das.plot()
das.fx_plot()

It seems that we have recorded a small event! However it looks like towards the end of the cable we have no useful data anymore. We will trim the record and focus on the channels 20 to 280

In [27]:
das.restrict_channels(20, 280)
das.total_channels # number of channels in the record
# print(das.channels_num) # all channel numbers

261

Now that we have indentified the channels that we are interested in, the next time when we are loading data, we can retrieve only the part of interest. This can decrease the loading time significantly. We do that by simply passing the channel range to the Fiber class: 
```
file = './example_data/2024_07_11_05h15m16s_HDAS_2DRawData_Strain.h5'
manufacturer = 'aragon'
channels = [20, 280]
das = Fiber(file, manufacturer, range_ch=channels)
```
We can also trim the record around the time of the event and normalize to the absolute max of the record before plotting:

In [28]:
t0 = das.start_time + 43
tf = t0 + 16
das.trim(t0, tf)
das.normalize() #default is abolute max normalization, other options are trace max, running absolute mean and 1bit
das.plot()

We can look at a few channels in more detail, lets go for 100 - 120:

In [29]:
section = (100, 120)
das.record_section(section)

We can plot a single channels waveform and the corresponding spectrogram and amplitude spectrum:

In [30]:
ch = 105
das.channel_plot(ch)

Render time: 691.3078 s


To explore the data a bit deeper we can call the data explorer:

In [14]:
das.view()

-----------------------------------------------------------------
Starting Fobench Data Viewer


IndexError: index 184 is out of bounds for axis 1 with size 178

IndexError: index 219 is out of bounds for axis 1 with size 178

IndexError: index 219 is out of bounds for axis 1 with size 178

IndexError: index 217 is out of bounds for axis 1 with size 178

...

IndexError: index 429 is out of bounds for axis 1 with size 178

IndexError: index 429 is out of bounds for axis 1 with size 178

IndexError: index 430 is out of bounds for axis 1 with size 178

IndexError: index 430 is out of bounds for axis 1 with size 178

-----------------------------------------------------------------
